# Examen Final – Diseño e Implementación de un Sistema de Recuperación de Información

**Nombre:** Alexis Bautista  
**Fecha de entrega:** 15 de julio de 2026

## A. Preparación del corpus

En esta sección se cargan y procesan los datos provenientes de la colección arXiv Paper Abstracts. El corpus original está dividido en dos archivos CSV (`arxiv_data_210930-054931.csv` y `arxiv_data.csv`) con ligeras variaciones en los nombres de sus columnas. 

Los pasos realizados son:
1. Lectura de ambos archivos.
2. Estandarización de las columnas (renombrar `summaries` a `abstracts`).
3. Concatenación de ambos DataFrames.
4. Eliminación de registros duplicados basados en el título del artículo.
5. Limpieza de valores nulos para asegurar la calidad del texto que será posteriormente vectorizado.

In [4]:
import pandas as pd

file1_path = 'corpus/arxiv_data_210930-054931.csv'
file2_path = 'corpus/arxiv_data.csv'

# 2. Cargar los datos
print("Cargando los archivos...")
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# Mostrar las columnas originales para verificación
print(f"Columnas df1: {df1.columns.tolist()}")
print(f"Columnas df2: {df2.columns.tolist()}")

# 3. Estandarizar nombres de columnas
# df1 tiene: ['terms', 'titles', 'abstracts']
# df2 tiene: ['titles', 'summaries', 'terms']
# Renombramos 'summaries' a 'abstracts' en df2 para que coincidan
df2 = df2.rename(columns={'summaries': 'abstracts'})

# Asegurarnos de que ambos tengan el mismo orden de columnas
column_order = ['titles', 'abstracts', 'terms']
df1 = df1[column_order]
df2 = df2[column_order]

# 4. Concatenar los DataFrames
print("\nConcatenando los datos...")
corpus_df = pd.concat([df1, df2], ignore_index=True)
print(f"Total de documentos antes de limpiar: {len(corpus_df)}")

# 5. Limpieza de datos (Eliminar duplicados y nulos)
# Eliminamos filas que no tengan título o abstract
corpus_df = corpus_df.dropna(subset=['titles', 'abstracts'])

# Eliminamos duplicados basados en el título del artículo
corpus_df = corpus_df.drop_duplicates(subset=['titles'], keep='first')

# Reiniciar el índice después de la limpieza
corpus_df = corpus_df.reset_index(drop=True)

print(f"Total de documentos después de limpiar: {len(corpus_df)}")
print("\nMuestra de los datos preparados:")
display(corpus_df.head())

Cargando los archivos...
Columnas df1: ['terms', 'titles', 'abstracts']
Columnas df2: ['titles', 'summaries', 'terms']

Concatenando los datos...
Total de documentos antes de limpiar: 107955
Total de documentos después de limpiar: 41212

Muestra de los datos preparados:


,titles,abstracts,terms
0,Multi-Level Attention Pooling for Graph Neural...,Graph neural networks (GNNs) have been widely ...,['cs.LG']
1,Decision Forests vs. Deep Networks: Conceptual...,Deep networks and decision forests (such as ra...,"['cs.LG', 'cs.AI']"
2,Power up! Robust Graph Convolutional Network v...,Graph convolutional networks (GCNs) are powerf...,"['cs.LG', 'cs.CR', 'stat.ML']"
3,Releasing Graph Neural Networks with Different...,With the increasing popularity of Graph Neural...,"['cs.LG', 'cs.CR']"
4,Recurrence-Aware Long-Term Cognitive Network f...,Machine learning solutions for pattern classif...,['cs.LG']


## B. Representación mediante embeddings

Para que el sistema de Recuperación de Información pueda realizar una búsqueda semántica, necesitamos representar los documentos del corpus como vectores densos (embeddings). 

En esta etapa:
1. Combinaremos el título y el abstract de cada artículo científico para enriquecer el contexto semántico de cada documento.
2. Utilizaremos un modelo de lenguaje preentrenado (en este caso, `all-MiniLM-L6-v2` mediante la librería `sentence-transformers`) por ser altamente eficiente y adecuado para representar oraciones y párrafos en inglés.
3. Generaremos los embeddings para todo el corpus preparado en el paso anterior.

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os

# Definir la ruta donde se guardarán los embeddings
embeddings_file = 'embeddings/embeddings_corpus.npy'

# 1. Cargar el modelo de embeddings
print("Cargando el modelo de embeddings...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Preparar el texto para vectorizar
# Combinamos título y abstract para tener una mejor representación del paper
print("Preparando el texto de los documentos...")
corpus_df['texto_completo'] = corpus_df['titles'] + ". " + corpus_df['abstracts']
textos_corpus = corpus_df['texto_completo'].tolist()

# 3. Generar o Cargar los embeddings
if os.path.exists(embeddings_file):
    print(f"\nArchivo '{embeddings_file}' encontrado.")
    print("Cargando embeddings desde el disco...")
    embeddings_corpus = np.load(embeddings_file)
else:
    print(f"\nGenerando embeddings para {len(textos_corpus)} documentos...")
    embeddings_corpus = embedding_model.encode(textos_corpus, show_progress_bar=True)
    
    print("Guardando embeddings generados en el disco...")
    np.save(embeddings_file, embeddings_corpus)

# 4. Verificación de las dimensiones
print("\n¡Embeddings listos exitosamente!")
print(f"Forma de la matriz de embeddings: {embeddings_corpus.shape}")
print(f"Total de documentos vectorizados: {embeddings_corpus.shape[0]}")
print(f"Dimensión de cada vector (embedding): {embeddings_corpus.shape[1]}")

Cargando el modelo de embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4187.07it/s]


Preparando el texto de los documentos...

Generando embeddings para 41212 documentos...


Batches: 100%|██████████| 1288/1288 [16:01<00:00,  1.34it/s]


Guardando embeddings generados en el disco...

¡Embeddings listos exitosamente!
Forma de la matriz de embeddings: (41212, 384)
Total de documentos vectorizados: 41212
Dimensión de cada vector (embedding): 384


## C. Almacenamiento y búsqueda vectorial

En esta etapa implementaremos el almacenamiento de los vectores y el mecanismo de búsqueda semántica. Para ello utilizaremos **FAISS**, una librería optimizada para la búsqueda de similitud de vectores densos.

Los pasos que realizaremos son:
1. Crear un índice vectorial de FAISS basado en la métrica de distancia L2 (Distancia Euclidiana), que es estándar para medir similitudes.
2. Cargar los embeddings (convertidos a formato `float32` que requiere FAISS) dentro de nuestro índice.
3. Crear una función auxiliar que tome una consulta en texto, la convierta a vector usando nuestro modelo de embeddings, y busque los "k" documentos más cercanos (vecinos más cercanos) en el índice FAISS.
4. Probaremos la función con una de las consultas de ejemplo.

In [7]:
import faiss
import numpy as np

# 1. Obtener la dimensión de los embeddings (384 para all-MiniLM-L6-v2)
dimension = embeddings_corpus.shape[1]

# 2. Crear el índice FAISS
print("Inicializando la base de datos vectorial (Índice FAISS)...")
# Usamos IndexFlatL2 para una búsqueda exacta basada en distancia euclidiana
index = faiss.IndexFlatL2(dimension)

# 3. Añadir los vectores al índice
# FAISS requiere estrictamente que los datos sean matrices numpy de tipo float32
embeddings_corpus_float32 = np.float32(embeddings_corpus)
index.add(embeddings_corpus_float32)

print(f"Total de vectores indexados exitosamente: {index.ntotal}")

# 4. Definir la función de búsqueda vectorial
def buscar_similares(query, k=5):
    """
    Busca los k documentos más similares a la consulta dada.
    Retorna una lista de diccionarios con el título, abstract y score (distancia).
    """
    # Vectorizar la consulta usando el mismo modelo
    query_embedding = embedding_model.encode([query])
    query_embedding = np.float32(query_embedding)
    
    # Buscar en el índice FAISS los k vecinos más cercanos
    # 'distancias' contiene los valores de similitud (menor es mejor)
    # 'indices' contiene las posiciones de los documentos en el corpus original
    distancias, indices = index.search(query_embedding, k)
    
    # Recuperar la información de los documentos correspondientes
    resultados = []
    for i, idx in enumerate(indices[0]):
        doc = corpus_df.iloc[idx]
        resultados.append({
            'score': distancias[0][i], # Distancia L2
            'title': doc['titles'],
            'abstract': doc['abstracts']
        })
        
    return resultados

# 5. Prueba de la búsqueda vectorial
# Utilizamos una de las consultas de ejemplo de los requerimientos
consulta_prueba = "What are the main applications of Graph Neural Networks?"
print(f"\nRealizando búsqueda de prueba para la consulta: '{consulta_prueba}'\n")

resultados_prueba = buscar_similares(consulta_prueba, k=3)

# Mostrar los resultados
for i, res in enumerate(resultados_prueba):
    print(f"--- Resultado {i+1} (Distancia: {res['score']:.4f}) ---")
    print(f"Título: {res['title']}")
    # Mostramos solo los primeros 250 caracteres del abstract para no saturar la pantalla
    print(f"Abstract: {res['abstract'][:250]}...\n")

Inicializando la base de datos vectorial (Índice FAISS)...
Total de vectores indexados exitosamente: 41212

Realizando búsqueda de prueba para la consulta: 'What are the main applications of Graph Neural Networks?'

--- Resultado 1 (Distancia: 0.6894) ---
Título: Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
Abstract: Graph neural networks denote a group of neural network models introduced for
the representation learning tasks on graph data specifically. Graph neural
networks have been demonstrated to be effective for capturing network structure
information, and t...

--- Resultado 2 (Distancia: 0.6941) ---
Título: Should Graph Neural Networks Use Features, Edges, Or Both?
Abstract: Graph Neural Networks (GNNs) are the first choice for learning algorithms on
graph data. GNNs promise to integrate (i) node features as well as (ii) edge
information in an end-to-end learning algorithm. How does this promise work out
practically? In ...

--- Re

## D. Recuperación

En esta sección se implementa la lógica principal de recuperación de información. 

El objetivo es:
1. Tomar los resultados brutos de la búsqueda vectorial (FAISS).
2. Aplicar un **umbral de similitud (distancia máxima)** para garantizar que los documentos recuperados realmente tengan relación con la consulta.
3. Formatear los textos recuperados en un solo bloque de "Contexto" que será inyectado posteriormente en el prompt del LLM.
4. Detectar si ningún documento supera el umbral de relevancia, cumpliendo con el requerimiento de indicar explícitamente cuando el corpus no contiene información suficiente para responder.

In [8]:
def recuperar_contexto(query, k=3, umbral_distancia=1.2):
    """
    Recupera los documentos relevantes y los formatea como contexto.
    Filtra los resultados usando un umbral de distancia L2 para asegurar relevancia.
    """
    # 1. Realizar la búsqueda usando la función definida en el Requerimiento C
    resultados_brutos = buscar_similares(query, k=k)
    
    contexto_texto = ""
    evidencias = []
    
    # 2. Filtrar y formatear los resultados
    for i, res in enumerate(resultados_brutos):
        # En distancia L2, valores más bajos significan mayor similitud.
        # Si la distancia es menor o igual al umbral, el documento es relevante.
        if res['score'] <= umbral_distancia:
            evidencias.append(res)
            contexto_texto += f"[Documento {len(evidencias)}]\n"
            contexto_texto += f"Título: {res['title']}\n"
            contexto_texto += f"Abstract: {res['abstract']}\n"
            contexto_texto += "-" * 50 + "\n"
            
    # 3. Manejo del caso donde no hay suficiente información
    if len(evidencias) == 0:
        return None, [] # Retornamos None para indicar falta de contexto
        
    return contexto_texto, evidencias

# --- Pruebas de la función de Recuperación ---

# Prueba 1: Una consulta que SÍ debería tener información en el corpus
consulta_valida = "Techniques for improving retrieval-augmented generation systems."
print(f"--- PRUEBA 1: Consulta válida ---\nQuery: '{consulta_valida}'")

contexto_1, evidencias_1 = recuperar_contexto(consulta_valida)

if contexto_1:
    print(f"Se recuperaron {len(evidencias_1)} documentos relevantes.")
    print("Muestra del contexto formateado (primeros 300 caracteres):")
    print(contexto_1[:300] + "...\n")
else:
    print("El corpus no contiene información suficiente para responder esta consulta.\n")


# Prueba 2: Una consulta fuera de dominio que NO debería pasar el umbral
consulta_invalida = "What is the best recipe to bake a chocolate cake?"
print(f"--- PRUEBA 2: Consulta fuera de dominio ---\nQuery: '{consulta_invalida}'")

# Ajustamos el umbral a un valor estricto para la prueba
contexto_2, evidencias_2 = recuperar_contexto(consulta_invalida, umbral_distancia=1.2)

if contexto_2:
    print(f"Se recuperaron {len(evidencias_2)} documentos relevantes.")
else:
    print("El corpus no contiene información suficiente para responder esta consulta.")

--- PRUEBA 1: Consulta válida ---
Query: 'Techniques for improving retrieval-augmented generation systems.'
Se recuperaron 3 documentos relevantes.
Muestra del contexto formateado (primeros 300 caracteres):
[Documento 1]
Título: Hybrid Retrieval-Generation Reinforced Agent for Medical Image Report Generation
Abstract: Generating long and coherent reports to describe medical images poses
challenges to bridging visual patterns with informative human linguistic
descriptions. We propose a novel Hybrid Retr...

--- PRUEBA 2: Consulta fuera de dominio ---
Query: 'What is the best recipe to bake a chocolate cake?'
El corpus no contiene información suficiente para responder esta consulta.


## E. Generación aumentada por recuperación

En esta sección integramos el Modelo de Lenguaje de Google (Gemini) para generar la respuesta final. 

El proceso es el siguiente:
1. Configuramos el acceso al LLM gestionando la API Key de Google de forma segura mediante variables de entorno y `getpass`, cumpliendo con el requerimiento de no exponerla en el código fuente.
2. Utilizamos el modelo `gemini-1.5-flash` e inyectamos un *System Instruction* que obligue al modelo a responder utilizando **únicamente** el contexto proporcionado.
3. Integramos la regla de validación: si la función de recuperación (paso D) determinó que no hay documentos relevantes, se omite llamar al LLM y se indica que no hay información suficiente.

In [ ]:
# Instalar la librería si no está instalada
# !pip install google-generativeai

import os
import getpass
import google.generativeai as genai

# 1. Gestión segura de la API Key
# getpass permite ingresar la clave sin dejar rastro en la celda
print("Por favor, ingresa tu API Key de Google (Gemini):")
os.environ["LLM_API_KEY"] = getpass.getpass()

# Configuramos la librería de Google con nuestra clave
genai.configure(api_key=os.environ.get("LLM_API_KEY"))

# 2. Función de Generación
def generar_respuesta_rag(query, contexto):
    """
    Genera una respuesta utilizando Gemini, basándose estrictamente en el contexto recuperado.
    """
    # Cumplimiento estricto: Indicar cuando no hay información suficiente
    if not contexto:
        return "Lo siento, el corpus actual no contiene información suficiente para responder a esta consulta."

    # Prompt del sistema para limitar las alucinaciones y forzar el uso del contexto
    system_prompt = """Eres un asistente académico experto en análisis de artículos científicos. 
    Tu tarea es responder a la pregunta del usuario utilizando ÚNICAMENTE la información provista en el "Contexto extraído de los artículos". 
    Si la respuesta no se encuentra en el contexto, debes indicar explícitamente que el corpus no tiene información suficiente para responder. 
    No utilices tu conocimiento previo ni inventes información."""

    # Inicializamos el modelo
    model = genai.GenerativeModel(
        model_name='gemini-2.5-flash',
        system_instruction=system_prompt,
        generation_config=genai.GenerationConfig(
            temperature=0.2 # Temperatura baja para mayor apego al texto
        )
    )

    user_message = f"Contexto extraído de los artículos:\n{contexto}\n\nPregunta del usuario: {query}"

    try:
        # Realizamos la llamada a la API
        response = model.generate_content(user_message)
        return response.text
    except Exception as e:
        return f"Error al generar la respuesta con Gemini: {str(e)}"

# --- Prueba del Pipeline Completo (Recuperación + Generación) ---

# Usamos otra de las consultas sugeridas
consulta_final = "How is reinforcement learning used in robotics?"
print(f"Consulta: '{consulta_final}'\n")

# A. Recuperar contexto 
print("1. Buscando documentos relevantes...")
contexto_recuperado, evidencias = recuperar_contexto(consulta_final)

# B. Generar respuesta
print("2. Generando respuesta con Gemini...")
respuesta_llm = generar_respuesta_rag(consulta_final, contexto_recuperado)

print("\n" + "="*50)
print("RESPUESTA DEL SISTEMA RAG:")
print("="*50)
print(respuesta_llm)

Por favor, ingresa tu API Key de Google (Gemini):
Consulta: 'How is reinforcement learning used in robotics?'

1. Buscando documentos relevantes...
2. Generando respuesta con Gemini...

RESPUESTA DEL SISTEMA RAG:
Según el contexto proporcionado:

El aprendizaje por refuerzo (RL) se utiliza en robótica para el aprendizaje autónomo en tareas de control y robótica reales. Los enfoques de RL se emplean para aprender controladores en sistemas reales como los robots. Además, las técnicas de aprendizaje por refuerzo basadas en modelos se aplican en tareas robóticas simuladas y reales, donde los modelos de transición y recompensa aprendidos se utilizan para la planificación. Los algoritmos de Deep Reinforcement Learning (DRL) también tienen implicaciones en el dominio de la robótica.


### Prueba Adicional: Consulta fuera de dominio

Esta celda comprueba que el sistema RAG cumple con la instrucción de indicar explícitamente cuando el corpus no contiene información suficiente para responder la consulta. Para ello, introducimos una pregunta sobre plomería doméstica, un tema que no debería existir en nuestra base de datos de inteligencia artificial y computación.

In [11]:
# Definimos una consulta completamente ajena al corpus
consulta_fuera_de_dominio = "How do I fix a leaking kitchen sink?"
print(f"Consulta: '{consulta_fuera_de_dominio}'\n")

# 1. Intentamos recuperar contexto
print("1. Buscando documentos relevantes...")
contexto_out, evidencias_out = recuperar_contexto(consulta_fuera_de_dominio)

# 2. Generamos la respuesta
print("2. Generando respuesta con Gemini...")
respuesta_out = generar_respuesta_rag(consulta_fuera_de_dominio, contexto_out)

print("\n" + "="*50)
print("RESPUESTA DEL SISTEMA RAG:")
print("="*50)
print(respuesta_out)

Consulta: 'How do I fix a leaking kitchen sink?'

1. Buscando documentos relevantes...
2. Generando respuesta con Gemini...

RESPUESTA DEL SISTEMA RAG:
Lo siento, el corpus actual no contiene información suficiente para responder a esta consulta.


## F. Presentación de evidencias

En esta sección se implementa un mecanismo para mostrar las fuentes utilizadas por el LLM. 

El objetivo es presentar no solo la respuesta generada, sino también los documentos recuperados (evidencias). Esto permite verificar la trazabilidad y la relación directa entre la consulta original, la información del corpus y la respuesta final elaborada por el modelo.

In [13]:
def presentar_resultados_con_evidencias(query, respuesta, evidencias):
    """
    Imprime de forma estructurada la consulta, la respuesta del LLM y las evidencias utilizadas.
    Permite verificar la relación entre los documentos recuperados y la respuesta.
    """
    print("=" * 60)
    print("SISTEMA RAG - RESPUESTA Y EVIDENCIAS")
    print("=" * 60)
    print(f"BÚSQUEDA: {query}\n")
    
    print("-" * 60)
    print("RESPUESTA GENERADA:")
    print("-" * 60)
    print(f"{respuesta}\n")
    
    print("-" * 60)
    print("EVIDENCIAS UTILIZADAS (FUENTES):")
    print("-" * 60)
    
    if not evidencias:
        print("No se encontraron evidencias suficientes en el corpus para esta consulta.")
    else:
        for i, doc in enumerate(evidencias):
            print(f"--- Documento {i+1} ---")
            print(f"Título: {doc['title']}")
            print(f"Similitud (Distancia L2): {doc['score']:.4f}")
            # Mostramos un fragmento representativo del abstract para facilitar la lectura
            fragmento = doc['abstract'][:300] + "..." if len(doc['abstract']) > 300 else doc['abstract']
            print(f"Abstract (Fragmento): {fragmento}\n")
            
    print("=" * 60)


# --- Prueba de la Presentación de Evidencias ---

# Reutilizamos los resultados obtenidos en el Requerimiento E para la consulta válida
# consulta_final = "How is reinforcement learning used in robotics?"
# Recordemos que las variables 'respuesta_llm' y 'evidencias' ya contienen los datos.

presentar_resultados_con_evidencias(consulta_final, respuesta_llm, evidencias)

SISTEMA RAG - RESPUESTA Y EVIDENCIAS
BÚSQUEDA: How is reinforcement learning used in robotics?

------------------------------------------------------------
RESPUESTA GENERADA:
------------------------------------------------------------
Según el contexto proporcionado:

El aprendizaje por refuerzo (RL) se utiliza en robótica para el aprendizaje autónomo en tareas de control y robótica reales. Los enfoques de RL se emplean para aprender controladores en sistemas reales como los robots. Además, las técnicas de aprendizaje por refuerzo basadas en modelos se aplican en tareas robóticas simuladas y reales, donde los modelos de transición y recompensa aprendidos se utilizan para la planificación. Los algoritmos de Deep Reinforcement Learning (DRL) también tienen implicaciones en el dominio de la robótica.

------------------------------------------------------------
EVIDENCIAS UTILIZADAS (FUENTES):
------------------------------------------------------------
--- Documento 1 ---
Título: Gaus

## G. Interfaz web conversacional

En esta sección se desarrolla una interfaz web interactiva utilizando la librería **Gradio**. 

La interfaz cumple con los siguientes requerimientos:
1. Permite el ingreso de consultas en lenguaje natural mediante un cuadro de texto.
2. Muestra de forma estructurada tanto la respuesta generada por el LLM como las evidencias (fragmentos de los documentos) utilizadas.
3. Permite al usuario realizar múltiples consultas de forma continua sin reiniciar la aplicación.
4. Procesa cada consulta de forma independiente, sin memoria conversacional (contexto histórico).

El diseño prioriza la claridad y la facilidad de uso, dividiendo la pantalla para leer la respuesta principal y verificar las fuentes simultáneamente.

In [14]:
import gradio as gr

def procesar_consulta_interfaz(query):
    """
    Función envolvente (wrapper) que conecta la interfaz con nuestro sistema RAG.
    Toma la consulta del usuario, recupera el contexto, genera la respuesta
    y formatea las evidencias para mostrarlas en la web.
    """
    # 1. Recuperar contexto (Usamos las funciones definidas en pasos anteriores)
    contexto_recuperado, evidencias = recuperar_contexto(query)
    
    # 2. Generar respuesta con el LLM
    respuesta_llm = generar_respuesta_rag(query, contexto_recuperado)
    
    # 3. Formatear las evidencias para mostrarlas en Markdown
    evidencias_md = "### Fuentes Utilizadas\n"
    if not evidencias:
        evidencias_md += "No se encontraron documentos relevantes en el corpus para esta consulta."
    else:
        for i, doc in enumerate(evidencias):
            # Usamos Markdown para darle un formato amigable a la evidencia
            evidencias_md += f"**Documento {i+1}: {doc['title']}**\n\n"
            evidencias_md += f"*Similitud (Distancia L2): {doc['score']:.4f}*\n\n"
            # Mostramos un fragmento del abstract
            fragmento = doc['abstract'][:300] + "..." if len(doc['abstract']) > 300 else doc['abstract']
            evidencias_md += f"> {fragmento}\n\n"
            evidencias_md += "---\n"
            
    return respuesta_llm, evidencias_md

# 4. Diseño de la Interfaz Web con Gradio Blocks
with gr.Blocks(theme=gr.themes.Soft(), title="Sistema RAG - arXiv") as demo:
    gr.Markdown("# Asistente de Recuperación Científica (RAG)")
    gr.Markdown("Ingresa tu consulta sobre inteligencia artificial, machine learning o ciencias de la computación. El sistema buscará en el corpus de **arXiv** y generará una respuesta basada en las evidencias encontradas.")
    
    with gr.Row():
        # Columna para la entrada del usuario
        with gr.Column(scale=1):
            entrada_usuario = gr.Textbox(
                label="Escribe tu consulta aquí", 
                placeholder="Ej. What are the main applications of Graph Neural Networks?", 
                lines=3
            )
            boton_enviar = gr.Button("Enviar Consulta", variant="primary")
            
    with gr.Row():
        # Columna principal para la respuesta del LLM
        with gr.Column(scale=2):
            salida_respuesta = gr.Markdown(label="Respuesta Generada")
            
        # Columna secundaria para mostrar las fuentes/evidencias
        with gr.Column(scale=1):
            salida_evidencias = gr.Markdown(label="Evidencias")
            
    # Conectar el botón con la función de procesamiento
    boton_enviar.click(
        fn=procesar_consulta_interfaz, 
        inputs=entrada_usuario, 
        outputs=[salida_respuesta, salida_evidencias]
    )
    
    # También permitir enviar presionando "Enter" en el cuadro de texto
    entrada_usuario.submit(
        fn=procesar_consulta_interfaz, 
        inputs=entrada_usuario, 
        outputs=[salida_respuesta, salida_evidencias]
    )

# 5. Lanzar la aplicación
# share=True genera un enlace público temporal (útil para pruebas externas)
# inline=True hace que la interfaz aparezca directamente debajo de esta celda
demo.launch(share=True, inline=True)

C:\Users\Asus\AppData\Local\Temp\ipykernel_3640\3953437521.py:32: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Sistema RAG - arXiv") as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
